# RAG Text2SQL Experiment

Notebook này thử nghiệm flow RAG hỗ trợ Text2SQL: retrieve few-shot SQL gần nhất từ `data/sql.json`, đưa các ví dụ đó vào prompt, sinh SQL an toàn, chạy database và tóm tắt kết quả.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# Setup path trước khi import ai_agent.* trong notebook.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "backend").exists():
    REPO_ROOT = Path("/home/bbsw/agent-pm")

BACKEND_ROOT = REPO_ROOT / "backend"
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

load_dotenv(REPO_ROOT / ".env")

DATA_DIR = BACKEND_ROOT / "ai_agent" / "text_to_sql" / "data"
FEWSHOT_PATH = DATA_DIR / "sql.json"

print("REPO_ROOT=", REPO_ROOT)
print("BACKEND_ROOT=", BACKEND_ROOT)
print("FEWSHOT_PATH=", FEWSHOT_PATH)
print("MODEL_NAME=", os.getenv("MODEL_NAME"))
print("BASE_URL=", os.getenv("BASE_URL"))

In [ ]:
import asyncio
import json
import re
import time
from typing import Any

import asyncpg
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from ai_agent.prompt.prompt import SCHEMA_COMPACT

SQL_SCHEMA = BACKEND_ROOT.parent / "init" / "init.sql"
schema = SQL_SCHEMA.read_text(encoding="utf-8") if SQL_SCHEMA.exists() else ""

_MUTATION_SQL = re.compile(
    r"\b(insert|update|delete|drop|alter|truncate|create|replace|merge|grant|revoke|vacuum|call|do)\b",
    re.IGNORECASE,
)
_NAMED_SQL_PLACEHOLDER = re.compile(r"(?<!:):[A-Za-z_][A-Za-z0-9_]*")

print("SCHEMA_COMPACT chars=", len(SCHEMA_COMPACT))
print("init.sql exists=", SQL_SCHEMA.exists())

## 1. Local Retriever

Retriever này đọc `sql.json`, vector hóa câu hỏi + SQL mẫu bằng TF-IDF, rồi chọn những ví dụ gần nhất với câu hỏi hiện tại.

In [ ]:
class FewShotSelector:
    """Select few-shot Text2SQL examples from sql.json using local TF-IDF similarity."""

    def __init__(self, json_path: str | Path = FEWSHOT_PATH):
        self.json_path = Path(json_path)
        self.examples = self._load_examples()
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
        corpus = [self._example_text(example) for example in self.examples]
        self.tfidf_matrix = self.vectorizer.fit_transform(corpus) if corpus else None

    def _load_examples(self) -> list[dict[str, str]]:
        if not self.json_path.exists() or self.json_path.stat().st_size == 0:
            return []

        data = json.loads(self.json_path.read_text(encoding="utf-8"))
        examples = []
        for item in data.get("FewShots", []):
            user_input = (item.get("input") or "").strip()
            query = (item.get("query") or "").strip()
            if user_input and query:
                examples.append({"input": user_input, "query": query})
        return examples

    @staticmethod
    def _example_text(example: dict[str, str]) -> str:
        return f"{example.get('input', '')}\n{example.get('query', '')}".strip()

    def select_examples(self, question: str, k: int = 3) -> list[dict[str, Any]]:
        if not self.examples or self.tfidf_matrix is None or k <= 0:
            return []

        query_vec = self.vectorizer.transform([question])
        scores = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        ranked_indexes = scores.argsort()[::-1][:k]

        selected = []
        for index in ranked_indexes:
            score = float(scores[index])
            if score <= 0:
                continue
            selected.append({**self.examples[index], "score": score})
        return selected

    @staticmethod
    def format_examples(examples: list[dict[str, Any]]) -> str:
        if not examples:
            return ""

        blocks = ["Ví dụ SQL liên quan đã retrieve từ knowledge base:"]
        for index, shot in enumerate(examples, start=1):
            blocks.append(
                f"Example {index} (similarity={shot.get('score', 0):.3f})\n"
                f"Câu hỏi: {shot['input']}\n"
                f"SQL:\n{shot['query']}"
            )
        return "\n\n".join(blocks)


selector = FewShotSelector()
print("fewshot_count=", len(selector.examples))
for example in selector.select_examples("Có bao nhiêu dự án đang chạy?", k=3):
    print(f"score={example['score']:.3f} input={example['input']}")

## 2. RAG Text2SQL Agent

`RAGText2SQLAgent` khác baseline ở bước `generate_sql`: trước khi gọi LLM, agent retrieve top-k few-shot examples rồi đưa vào prompt để LLM bắt chước pattern SQL phù hợp.

In [ ]:
class RAGText2SQLAgent:
    def __init__(
        self,
        db=None,
        llm: ChatOpenAI | None = None,
        top_k: int = 10,
        fewshot_k: int = 3,
        fewshot_path: str | Path = FEWSHOT_PATH,
        use_rag: bool = True,
    ):
        self.db = db
        self.top_k = top_k
        self.fewshot_k = fewshot_k
        self.use_rag = use_rag
        self.fewshot_selector = FewShotSelector(fewshot_path)
        self.last_examples: list[dict[str, Any]] = []
        self.last_timings: dict[str, float] = {}
        self.llm = llm or ChatOpenAI(
            model=os.getenv("MODEL_NAME"),
            timeout=60,
            api_key=os.getenv("API_KEY"),
            base_url=os.getenv("BASE_URL"),
        )

    def _schema_context(self) -> str:
        if self.db is not None and hasattr(self.db, "get_table_info"):
            return self.db.get_table_info()
        return SCHEMA_COMPACT

    def _clean_sql(self, sql: str) -> str:
        sql = sql.strip()
        if sql.startswith("```"):
            parts = sql.split("```")
            sql = parts[1] if len(parts) > 1 else sql
            if sql.lstrip().lower().startswith("sql"):
                sql = sql.lstrip()[3:]
        return sql.strip()

    def is_safe_sql(self, sql: str) -> bool:
        normalized = self._clean_sql(sql).strip()
        if not normalized.lower().startswith(("select", "with")):
            return False
        if not normalized.endswith(";"):
            return False
        if ";" in normalized.rstrip(";"):
            return False
        if _MUTATION_SQL.search(normalized):
            return False
        if _NAMED_SQL_PLACEHOLDER.search(normalized):
            return False
        return True

    def build_generate_sql_prompt(self, question: str, memory_context: str = "") -> str:
        retrieve_start = time.perf_counter()
        self.last_examples = self.fewshot_selector.select_examples(question, k=self.fewshot_k) if self.use_rag else []
        self.last_timings["retrieve"] = time.perf_counter() - retrieve_start

        examples_block = self.fewshot_selector.format_examples(self.last_examples)
        memory_block = f"\nNgữ cảnh hội thoại trước đó:\n{memory_context}\n" if memory_context else ""
        tenant_rule = "- Có bảng companies; chỉ JOIN/lọc theo company_id khi câu hỏi nhắc đến công ty cụ thể."

        return f"""
Bạn là một trợ lý AI chuyên Text2SQL cho hệ thống quản lý dự án.
Nhiệm vụ: tạo đúng 1 câu SQL PostgreSQL an toàn để trả lời câu hỏi người dùng.

Schema:
{self._schema_context()}
{memory_block}

{examples_block}

Quy tắc bắt buộc:
- Chỉ trả về SQL, không markdown, không giải thích.
- SQL phải bắt đầu bằng SELECT hoặc WITH và kết thúc bằng dấu chấm phẩy (;).
- Không dùng SELECT *.
- Không sinh placeholder như :project_id, :user_id, :limit.
- Không sinh lệnh ghi/xóa/sửa schema hoặc dữ liệu.
- Chỉ dùng bảng/cột có trong schema.
- Ưu tiên học pattern JOIN, enum cast, filter, ORDER BY, LIMIT từ các ví dụ retrieve ở trên khi phù hợp.
- Nếu câu hỏi không liên quan dữ liệu dự án/task/worklog/user/milestone, trả về: SELECT 'INVALID_QUESTION' AS message;
- Giới hạn kết quả danh sách khoảng {self.top_k} dòng nếu người dùng không yêu cầu số lượng khác.
{tenant_rule}

Câu hỏi người dùng:
{question}
"""

    async def generate_sql(self, question: str, memory_context: str = "") -> str:
        prompt_text = self.build_generate_sql_prompt(question, memory_context=memory_context)
        generate_start = time.perf_counter()
        response = await self.llm.ainvoke([
            SystemMessage(content=prompt_text),
            HumanMessage(content=f"Hãy tạo SQL cho câu hỏi: {question}"),
        ])
        self.last_timings["generate"] = time.perf_counter() - generate_start

        sql = self._clean_sql(response.content)
        if not self.is_safe_sql(sql):
            raise ValueError(f"Unsafe SQL generated: {sql}")

        print("RETRIEVED EXAMPLES")
        for example in self.last_examples:
            print(f"- score={example['score']:.3f} input={example['input']}")
        print("GENERATED SQL")
        print(sql)
        print(f"Retrieve time: {self.last_timings.get('retrieve', 0):.3f}s")
        print(f"Generate SQL time: {self.last_timings.get('generate', 0):.3f}s")
        return sql

    async def execute_sql(self, sql: str) -> list[dict]:
        conn = await asyncpg.connect(
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME"),
            host=os.getenv("DB_HOST", "localhost"),
            port=int(os.getenv("DB_PORT", 5432)),
        )
        try:
            rows = await conn.fetch(sql)
            return [dict(row) for row in rows]
        finally:
            await conn.close()

    async def summarize_result(self, question: str, rows: list[dict], memory_context: str = "") -> str:
        if not rows:
            return "Mình chưa tìm thấy dữ liệu phù hợp."

        summarize_start = time.perf_counter()
        memory_block = f"\nNgữ cảnh hội thoại trước đó:\n{memory_context}\n" if memory_context else ""
        prompt_text = f"""
Bạn là trợ lý quản lý dự án. Hãy trả lời câu hỏi của người dùng dựa trên kết quả truy vấn database.
{memory_block}

Câu hỏi:
{question}

Kết quả truy vấn dạng JSON:
{json.dumps(rows, ensure_ascii=False, default=str)}

Yêu cầu output:
- Chỉ trả về câu trả lời cuối cùng cho người dùng.
- Không hiển thị SQL, schema, tên bảng kỹ thuật, hoặc JSON thô.
- Trả lời ngắn gọn, tự nhiên bằng tiếng Việt.
"""
        response = await self.llm.ainvoke([
            SystemMessage(content=prompt_text),
            HumanMessage(content="Hãy trả lời người dùng dựa trên kết quả truy vấn ở trên."),
        ])
        self.last_timings["summarize"] = time.perf_counter() - summarize_start
        print(f"Summarize time: {self.last_timings['summarize']:.3f}s")
        return response.content.strip()

    async def execute(self, question: str, memory_context: str = "") -> dict:
        total_start = time.perf_counter()
        try:
            sql = await self.generate_sql(question, memory_context=memory_context)
        except ValueError as exc:
            return {"question": question, "sql": None, "result": str(exc), "retrieved_examples": self.last_examples}
        except Exception as exc:
            self.last_timings["total"] = time.perf_counter() - total_start
            return {
                "question": question,
                "sql": None,
                "result": str(exc),
                "answer": "Mình gặp lỗi khi gọi LLM để sinh SQL.",
                "retrieved_examples": self.last_examples,
                "timings": dict(self.last_timings),
            }

        execute_start = time.perf_counter()
        try:
            result = await self.execute_sql(sql)
        except Exception as exc:
            self.last_timings["execute"] = time.perf_counter() - execute_start
            self.last_timings["total"] = time.perf_counter() - total_start
            print(f"DB execution failed after {self.last_timings['execute']:.3f}s: {exc}", flush=True)
            return {
                "question": question,
                "sql": sql,
                "result": str(exc),
                "answer": "Mình đã tạo được truy vấn nhưng gặp lỗi khi chạy trên database. Bạn thử hỏi lại cụ thể hơn giúp mình nhé.",
                "retrieved_examples": self.last_examples,
                "timings": dict(self.last_timings),
            }

        self.last_timings["execute"] = time.perf_counter() - execute_start
        print(f"DB execution time: {self.last_timings['execute']:.3f}s")
        answer = await self.summarize_result(question, result, memory_context=memory_context)
        self.last_timings["total"] = time.perf_counter() - total_start

        return {
            "question": question,
            "sql": sql,
            "result": result,
            "answer": answer,
            "retrieved_examples": self.last_examples,
            "timings": dict(self.last_timings),
        }

## 3. Chạy một câu hỏi bằng RAG

In [ ]:
rag_agent = RAGText2SQLAgent(fewshot_k=3, use_rag=True)

question = "Có bao nhiêu dự án đang chạy?"
result = await rag_agent.execute(question)

print(json.dumps(result, ensure_ascii=False, default=str, indent=2))

## 4. So sánh Baseline vs RAG

`use_rag=False` vẫn dùng cùng schema và rules nhưng không đưa few-shot examples vào prompt. `use_rag=True` retrieve examples từ `sql.json`.

In [ ]:
async def compare_baseline_vs_rag(question: str, fewshot_k: int = 3):
    baseline = RAGText2SQLAgent(fewshot_k=0, use_rag=False)
    rag = RAGText2SQLAgent(fewshot_k=fewshot_k, use_rag=True)

    print("\n=== BASELINE: no retrieved examples ===")
    baseline_result = await baseline.execute(question)

    print("\n=== RAG: retrieved examples ===")
    rag_result = await rag.execute(question)

    summary = {
        "question": question,
        "baseline_sql": baseline_result.get("sql"),
        "rag_sql": rag_result.get("sql"),
        "rag_examples": [
            {"score": round(example["score"], 3), "input": example["input"]}
            for example in rag_result.get("retrieved_examples", [])
        ],
        "baseline_timings": baseline_result.get("timings"),
        "rag_timings": rag_result.get("timings"),
        "baseline_answer": baseline_result.get("answer"),
        "rag_answer": rag_result.get("answer"),
    }
    print("\n=== SUMMARY ===")
    print(json.dumps(summary, ensure_ascii=False, default=str, indent=2))
    return {"baseline": baseline_result, "rag": rag_result, "summary": summary}


comparison = await compare_baseline_vs_rag("Danh sách các task đang được thực hiện trong dự án CRM Thaco Go-Live Phase 1?", fewshot_k=3)

## 5. Batch Benchmark

In [ ]:
async def benchmark_questions(questions: list[str], fewshot_k: int = 3):
    agent = RAGText2SQLAgent(fewshot_k=fewshot_k, use_rag=True)
    rows = []

    for question in questions:
        print("\nQUESTION:", question)
        result = await agent.execute(question)
        rows.append({
            "question": question,
            "sql": result.get("sql"),
            "answer": result.get("answer"),
            "example_inputs": [example["input"] for example in result.get("retrieved_examples", [])],
            "timings": result.get("timings"),
        })

    return rows


questions = [
    "Có bao nhiêu dự án đang chạy?",
    "Danh sách các task đang được thực hiện trong dự án CRM Thaco Go-Live Phase 1?",
    "Ai là người quản lý dự án MTL?",
    "Dự án nào có deadline sắp tới nhất?",
    "Tổng số giờ đã được ghi lại cho dự án Vingroup?",
]

# Bỏ comment để chạy batch benchmark.
benchmark = await benchmark_questions(questions, fewshot_k=3)
print(json.dumps(benchmark, ensure_ascii=False, default=str, indent=2))

## 6. Đánh giá kết quả RAG

Cell này tổng hợp chất lượng RAG theo các tiêu chí thực dụng: retrieve có ví dụ liên quan không, SQL có sinh ra được không, execution có lỗi không, và latency từng bước.

In [ ]:
def evaluate_benchmark_results(rows: list[dict]) -> dict:
    if not rows:
        return {"message": "Chưa có benchmark để đánh giá. Hãy chạy cell benchmark trước."}

    total = len(rows)
    sql_generated = sum(1 for row in rows if row.get("sql"))
    db_success = sum(1 for row in rows if isinstance(row.get("answer"), str) and "gặp lỗi" not in row.get("answer", ""))
    has_retrieval = sum(1 for row in rows if row.get("example_inputs"))
    no_data = sum(1 for row in rows if row.get("answer") == "Mình chưa tìm thấy dữ liệu phù hợp.")

    timing_keys = ["retrieve", "generate", "execute", "summarize", "total"]
    avg_timings = {}
    for key in timing_keys:
        values = [row.get("timings", {}).get(key) for row in rows if row.get("timings", {}).get(key) is not None]
        avg_timings[key] = round(sum(values) / len(values), 3) if values else None

    retrieval_review = []
    for row in rows:
        retrieval_review.append({
            "question": row.get("question"),
            "top_example": (row.get("example_inputs") or [None])[0],
            "sql_preview": (row.get("sql") or "")[:180],
            "answer": row.get("answer"),
        })

    return {
        "total_questions": total,
        "sql_generated_rate": round(sql_generated / total, 3),
        "db_success_rate": round(db_success / total, 3),
        "retrieval_coverage": round(has_retrieval / total, 3),
        "no_data_count": no_data,
        "avg_timings_seconds": avg_timings,
        "retrieval_review": retrieval_review,
        "notes": [
            "RAG tốt khi top_example cùng intent với câu hỏi, ví dụ count project, deadline, worklog.",
            "Nếu top_example lệch intent, cần bổ sung few-shot vào sql.json cho intent đó.",
            "Câu trả lời 'chưa tìm thấy dữ liệu' không nhất thiết là sai SQL; có thể database không có record khớp filter.",
        ],
    }


evaluation = evaluate_benchmark_results(benchmark if "benchmark" in globals() else [])
print(json.dumps(evaluation, ensure_ascii=False, default=str, indent=2))